# Analyse du Data Drift

Comparaison des données d’entraînement avec un groupe de clients de production aux revenus élevés, à l’aide d’Evidently.


In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from evidently import Report
from evidently.presets import DataDriftPreset

## Chargement des données

Les données de référence viennent du fichier CSV et les données de production sont stockées dans PostgreSQL.

In [2]:
if os.path.exists("/.dockerenv"):
    load_dotenv("docker.env")
else:
    load_dotenv("local.env")

df_train = pd.read_csv("df_total_clean.csv")
engine = create_engine(os.getenv("URLBDD"))
df_production = pd.read_sql('SELECT * FROM "predictions_inputs"', con=engine)

df_train = df_train.drop(columns=["SK_ID_CURR"])
df_production = df_production.drop(columns=["SK_ID_CURR"])

## Simulation du drift

Je garde uniquement les 25 % de clients avec les revenus les plus élevés en production pour simuler une différence avec les données d’entraînement.

In [3]:
seuil = df_production["AMT_INCOME_TOTAL"].quantile(0.75)
clients_riches = df_production[df_production["AMT_INCOME_TOTAL"] > seuil]
df_train_sample = df_train.sample(n=20000, random_state=42)

print("Clients sélectionnés :", len(clients_riches))

Clients sélectionnés : 19


## Rapport Evidently

Evidently compare les distributions des variables entre les deux jeux de données.

In [4]:
report = Report(metrics=[DataDriftPreset()])
resultat = report.run(reference_data=df_train_sample, current_data=clients_riches)
resultat.save_html("drift_report.html")

c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning:

invalid value encountered in divide

c:\Users\Tristan\Desktop\VSC\GlobalEnv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning:

invalid value encountered in divide



## Résultats — comparaison des profils

**65 variables sur 117 présentent un drift (55,6 %)**.

| Indicateur | Train | Production Filtrée | Écart | Drift détecté |
|:--|--:|--:|--:|:--:|
| **Annuité du crédit** (`AMT_ANNUITY`) | ≈ 27 100 | ≈ 35 500 | **+31 %** | Oui |
| **Montant du crédit** (`AMT_CREDIT_y`) | ≈ 615 300 | ≈ 823 700 | **+34 %** | Oui |
| **Prix du bien** (`AMT_GOODS_PRICE`) | ≈ 579 600 | ≈ 731 300 | **+26 %** | Oui |
| **Études supérieures** (`Higher_education`) | 24,3 % | 42,1 % (8/19) | **+17,8 points** | Oui |
| **Femmes** (`F`) | 66,3 % | 31,6 % (6/19) | **−34,7 points** | Oui |
| **Séparés** (`Separated`) | 6,4 % | 21,1 % (4/19) | **+14,6 points** | Oui |
| **Cible `TARGET = 1`** | 8,2 % | 5,3 % (1/19) | −3,0 points | Non |

### Interprétation

Le filtre sur les **25 % de revenus les plus élevés** crée volontairement un groupe différent de la population générale. Dans ce groupe, les **crédits et les annuités sont environ 30 % plus élevés**, et la part de clients ayant fait des **études supérieures passe de 24 % à 42 %**.